In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict

from mad.objs.constants import G0, titan_stage_1, titan_stage_2
from mad.objs.common_schemas import MovableObject
from mad.objs.projectiles import ProjectileConfig, Projectile
from mad.logger import SourceLogger

logger = SourceLogger()

In [9]:
class Payload(MovableObject):
    mass: float # kg
    area: float # m^2
    yield_kt: float # kt


@dataclass
class StageConfig:
    dry_mass: float  # kg
    propellant_mass: float  # kg
    thrust: float # N / s
    Isp: float  # s
    area: float # m^2
    Cd: float
    time_ECO: float # s
    time_sep: float  # s
    payload: Payload | None = None
    name: str = "Stage"

    @property
    def to_dict(self):
        return asdict(self)

In [14]:
class MissileStage:
    def __init__(self, cfg: StageConfig):
        self.config = cfg
        self.dry_mass = cfg.dry_mass
        self.propellant_mass = cfg.propellant_mass

        self.thrust = cfg.thrust
        self.Isp = cfg.Isp

        self.area = cfg.area
        self.Cd = cfg.Cd

        self.exhaust_velocity = cfg.Isp * G0
        self.mass_flow_rate = cfg.thrust / self.exhaust_velocity

        self.active: bool = True
        self.payload = cfg.payload
        self.name = cfg.name

    @property
    def mass(self)->float:
        payload_mass = self.payload.mass if self.payload else 0.0
        return self.dry_mass + self.propellant_mass + payload_mass

    def update(self, dt: float)->None:
        if not self.active:
            return

        if self.propellant_mass > 0:
            dm = self.mass_flow_rate * dt
            self.propellant_mass = max(0.0, self.propellant_mass - dm)
        else:
            logger["Missile"].info(f"{self.name} ran out of propellant.")
            self.active = False

    def thrust_force(self, direction: NDArray) -> NDArray:
        if self.propellant_mass > 0:
            return self.thrust * direction
        return np.zeros_like(direction)

In [ ]:
@dataclass
class MissileConfig:
    stages: list[MissileStage]
    position: list[float]
    name: str = "MultiStageMissile"
    guidance = None

    
    @property
    def to_dict(self):
        return asdict(self)

class Missile(MovableObject):
    def __init__(self, cfg: MissileConfig):
        super().__init__(position=cfg.position, name = cfg.name)

        self.stages = cfg.stages
        self.guidance = cfg.guidance

    @property
    def mass(self):
        return sum(stage.mass for stage in self.stages)

    @property
    def area(self):
        return sum(stage.area for stage in self.stages)
    
    def get_guidance(self)->NDArray:
        return np.zeros_like(self.position)
    
    def step(self, dt):
        self.thrust = np.zeros_like(self.position)
        self.update(dt)
        return
    
    def update(self, dt: float) -> None | Projectile:
        running_stage = self.stages[0]
        running_stage.update(dt)
        
        direction = self.get_guidance()
        self.thrust_force = running_stage.thrust_force(direction)

        if not running_stage.active:
            stage_cfg = ProjectileConfig(position = self.position.tolist(), 
                                            velocity = self.velocity.tolist(),
                                            mass = running_stage.dry_mass,
                                            name = running_stage.name,
                                            area = running_stage.area,
                                            Cd = running_stage.Cd,
                                            )

            del(self.stages[0])
            logger["Missile"].info(f"{self.name} - {running_stage.name} separated.")
            if len(self.stages) == 0: 
                self.active = False
                logger["Missile"].info(f"{self.name} inactivated.")
            return Projectile(stage_cfg)

In [31]:
stage1 = MissileStage(StageConfig(**titan_stage_1))
stage2 = MissileStage(StageConfig(**titan_stage_2))

titan = Missile(MissileConfig(position = [0, 0], stages = [stage1, stage2], name="Titan I"))

In [32]:
dt = 1.0
active_objs = [titan]
inactive_objs = []
for t in range(500):
    for obj in active_objs:
        if not obj.active: 
            continue
        result = obj.step(dt)
        if result is not None:
            active_objs.append(result)
            logger["Simulation"].info(f"{result.name} added to Simulation.")

TypeError: Missile.step() missing 1 required positional argument: 'thrust'